# 04 · Ratio Statistic: Unfolding-Free Local Comparison

This notebook studies the **ratio statistic** for consecutive zeta-zero spacings.

Instead of unfolding the spacing sequence, we compare adjacent gaps directly.

## Why this matters

If
\[
g_n = t_{n+1} - t_n,
\]
then a standard local ratio is
\[
r_n = \frac{g_{n+1}}{g_n}.
\]

A commonly used symmetrized version is
\[
\tilde r_n = \frac{\min(g_n,g_{n+1})}{\max(g_n,g_{n+1})},
\qquad 0 \le \tilde r_n \le 1.
\]

The advantage is that \(\tilde r_n\) is **unfolding-free**:
it compares neighboring gaps locally, so it is less sensitive to slow changes in the average spacing with height.

This notebook compares:

1. zeta-zero gap ratios
2. Poisson / exponential controls
3. GUE-style controls via the Wigner surmise

It also connects \(\tilde r_n\) to the local three-zero neighborhood statistic already used in the repo.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Collect nontrivial zeta zeros and raw gaps

In [ ]:
N = 400
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
gaps = np.diff(t)

len(t), len(gaps), gaps[:5]

## Build control spacing sequences

### Poisson
Independent exponential spacings with mean 1.

### GUE-style reference
A Wigner-surmise-based sample used as a GUE-style proxy.

> **Note:** The GUE comparison here uses the Wigner surmise as a proxy, not full random-matrix simulation.

In [ ]:
M = len(gaps)

poisson_gaps = rng.exponential(scale=1.0, size=M)

def wigner_gue_pdf(s):
    return (32 / np.pi**2) * s**2 * np.exp(-4 * s**2 / np.pi)

def sample_wigner_gue(size, rng, s_max=3.5):
    grid = np.linspace(0, s_max, 2000)
    pdf_max = wigner_gue_pdf(grid).max()
    out = []
    while len(out) < size:
        s = rng.uniform(0, s_max, size=size)
        u = rng.uniform(0, pdf_max, size=size)
        accepted = s[u <= wigner_gue_pdf(s)]
        out.extend(accepted.tolist())
    return np.array(out[:size])

gue_gaps = sample_wigner_gue(M, rng)

poisson_gaps.mean(), gue_gaps.mean()

## Define ratio statistics

Given a spacing sequence \(g_n\), define:

- raw ratio:
  \[
  r_n = \frac{g_{n+1}}{g_n}
  \]
- symmetrized ratio:
  \[
  \tilde r_n = \frac{\min(g_n,g_{n+1})}{\max(g_n,g_{n+1})}
  \]

The symmetrized ratio always lies in \([0,1]\).

In [ ]:
def ratio_stats(seq):
    g1 = seq[:-1]
    g2 = seq[1:]
    r = g2 / g1
    r_tilde = np.minimum(g1, g2) / np.maximum(g1, g2)
    return r, r_tilde

zeta_r, zeta_r_tilde = ratio_stats(gaps)
poisson_r, poisson_r_tilde = ratio_stats(poisson_gaps)
gue_r, gue_r_tilde = ratio_stats(gue_gaps)

zeta_r_tilde.mean(), poisson_r_tilde.mean(), gue_r_tilde.mean()

## First look at the zeta ratio series

In [ ]:
index = np.arange(1, len(zeta_r) + 1)

plt.figure(figsize=(8.4, 4.8))
plt.plot(index, zeta_r, marker="o", linewidth=1)
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("ratio index n")
plt.ylabel(r"$r_n = g_{n+1}/g_n$")
plt.title("Raw ratio statistic for zeta gaps")
plt.show()

In [ ]:
plt.figure(figsize=(8.4, 4.8))
plt.plot(index, zeta_r_tilde, marker="o", linewidth=1)
plt.axhline(zeta_r_tilde.mean(), linestyle="--", linewidth=1, label=f"mean ≈ {zeta_r_tilde.mean():.3f}")
plt.xlabel("ratio index n")
plt.ylabel(r"$\tilde r_n$")
plt.title("Symmetrized ratio statistic for zeta gaps")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

## Histogram comparison for \(\tilde r_n\)

This is the main unfolding-free comparison in the notebook.

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 1, 26)
plt.hist(zeta_r_tilde, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_r_tilde, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_r_tilde, bins=bins, density=True, alpha=0.45, label="GUE (Wigner)")
plt.xlabel(r"$\tilde r_n = \min(g_n,g_{n+1})/\max(g_n,g_{n+1})$")
plt.ylabel("density")
plt.title("Symmetrized ratio comparison")
plt.legend()
plt.show()

## Small-\(\tilde r\) region

Small values of \(\tilde r_n\) indicate a very uneven local pair of neighboring gaps.

In [ ]:
small_bins = np.linspace(0, 0.5, 18)

plt.figure(figsize=(8.8, 5.0))
plt.hist(zeta_r_tilde, bins=small_bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_r_tilde, bins=small_bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_r_tilde, bins=small_bins, density=True, alpha=0.45, label="GUE (Wigner)")
plt.xlabel(r"$\tilde r_n$")
plt.ylabel("density")
plt.title("Small-ratio comparison")
plt.legend()
plt.show()

## Compare with the repo's local balance statistic

Previously we used
\[
b_n = \frac{\min(g_L,g_R)}{\max(g_L,g_R)}.
\]

This is algebraically the same form as \(\tilde r_n\), just indexed as a local three-zero neighborhood.
So in this notebook:

- ratio language = standard statistical comparison
- local balance language = repo-specific local neighborhood framing

In [ ]:
def local_balance(seq):
    left = seq[:-1]
    right = seq[1:]
    return np.minimum(left, right) / np.maximum(left, right)

zeta_balance = local_balance(gaps)
poisson_balance = local_balance(poisson_gaps)
gue_balance = local_balance(gue_gaps)

np.allclose(zeta_r_tilde, zeta_balance), np.allclose(poisson_r_tilde, poisson_balance), np.allclose(gue_r_tilde, gue_balance)

## Overlay means

This gives a compact summary of the three ensembles.

In [ ]:
labels = ["zeta", "Poisson", "GUE"]
means = [
    zeta_r_tilde.mean(),
    poisson_r_tilde.mean(),
    gue_r_tilde.mean(),
]

plt.figure(figsize=(7.0, 4.6))
plt.bar(labels, means)
plt.ylabel(r"mean symmetrized ratio $\langle \tilde r \rangle$")
plt.title("Mean unfolding-free ratio by ensemble")
plt.show()

means

## Raw-ratio histogram (truncated view)

The raw ratio \(r_n=g_{n+1}/g_n\) can have a long tail, so we truncate the plot range for readability.

In [ ]:
zeta_r_clip = zeta_r[(zeta_r >= 0) & (zeta_r <= 4)]
poisson_r_clip = poisson_r[(poisson_r >= 0) & (poisson_r <= 4)]
gue_r_clip = gue_r[(gue_r >= 0) & (gue_r <= 4)]

plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 4, 30)
plt.hist(zeta_r_clip, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_r_clip, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_r_clip, bins=bins, density=True, alpha=0.45, label="GUE (Wigner)")
plt.xlabel(r"$r_n = g_{n+1}/g_n$")
plt.ylabel("density")
plt.title("Raw ratio comparison (truncated at 4)")
plt.legend()
plt.show()

## Quantitative summaries

In [ ]:
def summary_stats(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

summary = {
    "zeta_r_tilde": summary_stats(zeta_r_tilde),
    "poisson_r_tilde": summary_stats(poisson_r_tilde),
    "gue_r_tilde": summary_stats(gue_r_tilde),
    "zeta_r": summary_stats(zeta_r),
    "poisson_r": summary_stats(poisson_r),
    "gue_r": summary_stats(gue_r),
}
summary

## Notes

- The symmetrized ratio \(\tilde r_n\) is unfolding-free and is often more stable than direct spacing histograms when the mean spacing varies slowly with height.
- In this repo, the local three-zero balance statistic is the same algebraic object as the symmetrized ratio.
- Poisson controls tend to put more weight near very small \(\tilde r\), corresponding to more uneven neighboring gaps.
- Zeta and GUE-style references are expected to look closer to each other than either does to Poisson.

## Next directions

A natural next notebook could do one of these:

1. compare larger zeta-zero samples  
2. generate full random Hermitian matrices instead of using the Wigner surmise  
3. compute pair-correlation-style quantities  
4. study longer local patterns using 4-gap or 5-gap neighborhood descriptors